# Jordan-Wide Basin-Specific 3-Model Ensemble — SSP5-8.5
## Notebook 19

Generates six Jordan-wide NetCDF files (one per time period) using the basin-specific top-3 model ensemble for SSP5-8.5. Logic identical to Notebook 05; only the model directory and output path differ.

| Period label | Years |
|---|---|
| 1961_2070 | Full simulation period |
| 1995_2014 | Reference period |
| 2015_2020 | Transitional |
| 2021_2040 | Near-future |
| 2041_2060 | Mid-future |
| 2061_2070 | End-of-century |

**Output:** `ssp5-8.5 output\3 ensemble models\Jordan_3model_ensemble_ssp585_{period}.nc`

## 1. Import Libraries

In [1]:
import warnings
import datetime
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree
from shapely.geometry import Point
import time as _time

warnings.filterwarnings("ignore")

print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr), ("geopandas", gpd)]:
    print(f"  {lib:<12}: {mod.__version__}")

Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0
  geopandas   : 1.0.1


## 2. Configuration

In [2]:
SHAPEFILE    = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                    r"\jordan.basins\surface.basins.for.jordan\Jo.shp")

RANKINGS_CSV = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                    r"\validation\single.model\basin_model_rankings.csv")

# SSP5-8.5 model NC files (filename prefix: merged_)
MODEL_DIR    = Path(r"D:\RICAAR8.5\merge\Precipitation")

OUTPUT_DIR   = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                    r"\ssp5-8.5 output\3 ensemble models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    "CMCC-CM2-SR5",
    "CNRM-ESM2-1",
    "EC-Earth3-Veg",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-LR",
    "NorESM2-MM",
]

PR_VAR       = "prAdjust"
SYRIA_BASINS = {"YARMOUK (SYRIA)", "AZRAQ (SYRIA)", "AMMAN ZARQA (SYRIA)"}

BASIN_NAME_MAP = {
    "HAMMAD"                    : "HAMMAD",
    "YARMOUK (JORDAN)"          : "YARMOUK (JORDAN)",
    "J.VALLEY-YARMOUK TRIANGLE" : None,
    "JORDAN VALLY (JORDAN)"     : "JORDAN VALLY (JORDAN)",
    "N.R.S.W"                   : "N.R.S.W",
    "AZRAQ (JORDAN)"            : "AZRAQ (JORDAN)",
    "AMMAN ZARQA (JORDAN)"      : "AMMAN ZARQA (JORDAN)",
    "S.R.S.W"                   : "S.R.S.W",
    "MUJIB"                     : "MUJIB",
    "D.S.R.S.W"                 : "D.S.R.S.W",
    "W. ARABA NORTH"            : "W. ARABA NORTH",
    "HASA"                      : "HASA",
    "JAFER"                     : "JAFER",
    "WADI ARABA SOUTH"          : None,
    "QA DISI & SOUTHERN DESERT" : None,
    "SARHAN"                    : None,
}

PERIODS = [
    ("1961_2070", 1961, 2070),
    ("1995_2014", 1995, 2014),
    ("2015_2020", 2015, 2020),
    ("2021_2040", 2021, 2040),
    ("2041_2060", 2041, 2060),
    ("2061_2070", 2061, 2070),
]

print("Configuration loaded.")
print(f"  Model dir  : {MODEL_DIR}")
print(f"  Shapefile  : {SHAPEFILE}")
print(f"  Output dir : {OUTPUT_DIR}")
print()
print("Output periods:")
for label, y0, y1 in PERIODS:
    print(f"  {label}  ({y0}–{y1})")

Configuration loaded.
  Model dir  : D:\RICAAR8.5\merge\Precipitation
  Shapefile  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\jordan.basins\surface.basins.for.jordan\Jo.shp
  Output dir : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ssp5-8.5 output\3 ensemble models

Output periods:
  1961_2070  (1961–2070)
  1995_2014  (1995–2014)
  2015_2020  (2015–2020)
  2021_2040  (2021–2040)
  2041_2060  (2041–2060)
  2061_2070  (2061–2070)


## 3. Load Shapefile and Rankings

In [3]:
gdf_all    = gpd.read_file(SHAPEFILE)
gdf_jordan = gdf_all[~gdf_all["BASIN_NAME"].isin(SYRIA_BASINS)].copy().reset_index(drop=True)

print(f"Total shapefile features : {len(gdf_all)}")
print(f"Jordan-only retained     : {len(gdf_jordan)}")
print()
print("Jordan basins:")
for _, row in gdf_jordan.iterrows():
    print(f"  {row['BASIN_NAME']}")

rankings = pd.read_csv(RANKINGS_CSV)
rankings.columns = [c.strip() for c in rankings.columns]

top3_per_basin = {}
for basin, grp in rankings.groupby("Basin"):
    top3_per_basin[basin] = grp.sort_values("Avg_Rank")["Model"].iloc[:3].tolist()

print()
print("Top-3 models per validated basin:")
print(f"{'Basin':<28} {'Rank-1':<22} {'Rank-2':<22} {'Rank-3'}")
print("-" * 88)
for basin, models in top3_per_basin.items():
    print(f"{basin:<28} {'  |  '.join(models)}")

Total shapefile features : 19
Jordan-only retained     : 16

Jordan basins:
  HAMMAD
  YARMOUK (JORDAN)
  J.VALLEY-YARMOUK TRIANGLE
  JORDAN VALLY (JORDAN)
  N.R.S.W
  AZRAQ (JORDAN)
  AMMAN ZARQA (JORDAN)
  S.R.S.W
  MUJIB
  D.S.R.S.W
  W. ARABA NORTH
  HASA
  JAFER
  WADI ARABA SOUTH
  QA DISI & SOUTHERN DESERT
  SARHAN

Top-3 models per validated basin:
Basin                        Rank-1                 Rank-2                 Rank-3
----------------------------------------------------------------------------------------
AMMAN ZARQA (JORDAN)         MPI-ESM1-2-LR  |  NorESM2-MM  |  CMCC-CM2-SR5
AZRAQ (JORDAN)               MPI-ESM1-2-LR  |  NorESM2-MM  |  CMCC-CM2-SR5
D.S.R.S.W                    IPSL-CM6A-LR  |  MPI-ESM1-2-LR  |  NorESM2-MM
HAMMAD                       MPI-ESM1-2-LR  |  CMCC-CM2-SR5  |  NorESM2-MM
HASA                         MPI-ESM1-2-LR  |  CMCC-CM2-SR5  |  CNRM-ESM2-1
JAFER                        NorESM2-MM  |  MPI-ESM1-2-LR  |  CMCC-CM2-SR5
JORDAN VALLY (JORDA

## 4. Assign Top-3 Models to All Jordan Basins (KD-Tree for Unranked)

In [4]:
gdf_jordan["centroid_lon"] = gdf_jordan.geometry.centroid.x
gdf_jordan["centroid_lat"] = gdf_jordan.geometry.centroid.y

ranked_mask = gdf_jordan["BASIN_NAME"].apply(
    lambda n: BASIN_NAME_MAP.get(n) is not None
              and BASIN_NAME_MAP.get(n) in top3_per_basin
)
ranked_gdf   = gdf_jordan[ranked_mask].copy()
unranked_gdf = gdf_jordan[~ranked_mask].copy()

ranked_coords = ranked_gdf[["centroid_lat", "centroid_lon"]].values
tree = cKDTree(ranked_coords)

basin_assignment = {}
for _, row in gdf_jordan.iterrows():
    shp_name     = row["BASIN_NAME"]
    ranking_name = BASIN_NAME_MAP.get(shp_name)

    if ranking_name is not None and ranking_name in top3_per_basin:
        top3   = top3_per_basin[ranking_name]
        source = "direct"
    else:
        query        = [row["centroid_lat"], row["centroid_lon"]]
        dist, idx    = tree.query(query)
        nearest_shp  = ranked_gdf.iloc[idx]["BASIN_NAME"]
        nearest_rank = BASIN_NAME_MAP[nearest_shp]
        top3         = top3_per_basin[nearest_rank]
        source       = f"KD-tree -> {nearest_shp} ({dist * 111.32:.1f} km)"

    basin_assignment[shp_name] = {"top3": top3, "source": source}

print(f"{'Basin':<32} {'Assignment source':<38} {'Top-3 Models'}")
print("-" * 105)
for b, info in basin_assignment.items():
    print(f"{b:<32} {info['source']:<38} {' | '.join(info['top3'])}")

Basin                            Assignment source                      Top-3 Models
---------------------------------------------------------------------------------------------------------
HAMMAD                           direct                                 MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
YARMOUK (JORDAN)                 direct                                 MPI-ESM1-2-LR | NorESM2-MM | CNRM-ESM2-1
J.VALLEY-YARMOUK TRIANGLE        KD-tree -> N.R.S.W (29.2 km)           NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
JORDAN VALLY (JORDAN)            direct                                 MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
N.R.S.W                          direct                                 NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
AZRAQ (JORDAN)                   direct                                 MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
AMMAN ZARQA (JORDAN)             direct                                 MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
S.R.S.W         

## 5. Load Grid and Build Basin-ID Grid

In [5]:
sample_nc = MODEL_DIR / f"merged_{MODELS[0]}.nc"
with xr.open_dataset(sample_nc) as ds:
    lats         = ds["lat"].values
    lons         = ds["lon"].values
    all_times    = ds["time"].values
    sample_attrs = dict(ds[PR_VAR].attrs)
    global_attrs = dict(ds.attrs)

n_lat, n_lon = len(lats), len(lons)
print(f"Grid: {n_lat} lat × {n_lon} lon  "
      f"({lats.min():.2f}–{lats.max():.2f}°N, {lons.min():.2f}–{lons.max():.2f}°E)")
print(f"Time: {len(all_times):,} steps  "
      f"({pd.Timestamp(all_times[0]).date()} → {pd.Timestamp(all_times[-1]).date()})")
print()

jordan_basin_names = gdf_jordan["BASIN_NAME"].tolist()
basin_id_grid      = np.full((n_lat, n_lon), -1, dtype=np.int8)

t_start = _time.time()
for li, lat in enumerate(lats):
    for lj, lon in enumerate(lons):
        pt = Point(lon, lat)
        for bi, bname in enumerate(jordan_basin_names):
            if gdf_jordan.iloc[bi].geometry.contains(pt):
                basin_id_grid[li, lj] = bi
                break

elapsed = _time.time() - t_start
n_jordan_cells  = int((basin_id_grid >= 0).sum())
n_outside_cells = int((basin_id_grid < 0).sum())
print(f"Grid cell classification complete  ({elapsed:.0f}s)")
print(f"  Inside Jordan  : {n_jordan_cells:,} cells")
print(f"  Outside Jordan : {n_outside_cells:,} cells")
print()
print("Grid cells per basin:")
for bi, bname in enumerate(jordan_basin_names):
    n = int((basin_id_grid == bi).sum())
    print(f"  {bname:<32} : {n:>4} cells")

Grid: 85 lat × 75 lon  (27.05–35.45°N, 33.55–40.95°E)
Time: 40,177 steps  (1961-01-01 → 2070-12-31)

Grid cell classification complete  (3s)
  Inside Jordan  : 840 cells
  Outside Jordan : 5,535 cells

Grid cells per basin:
  HAMMAD                           :  176 cells
  YARMOUK (JORDAN)                 :   11 cells
  J.VALLEY-YARMOUK TRIANGLE        :    0 cells
  JORDAN VALLY (JORDAN)            :    5 cells
  N.R.S.W                          :   11 cells
  AZRAQ (JORDAN)                   :  113 cells
  AMMAN ZARQA (JORDAN)             :   33 cells
  S.R.S.W                          :    6 cells
  MUJIB                            :   61 cells
  D.S.R.S.W                        :   15 cells
  W. ARABA NORTH                   :   27 cells
  HASA                             :   22 cells
  JAFER                            :  116 cells
  WADI ARABA SOUTH                 :   55 cells
  QA DISI & SOUTHERN DESERT        :   36 cells
  SARHAN                           :  153 cells


## 6. Build Per-Cell Model Mask

In [6]:
model_cell_mask = {m: np.zeros((n_lat, n_lon), dtype=bool) for m in MODELS}
n_models_per_cell = np.zeros((n_lat, n_lon), dtype=np.int8)

for li in range(n_lat):
    for lj in range(n_lon):
        bi = basin_id_grid[li, lj]
        if bi < 0:
            continue
        bname = jordan_basin_names[bi]
        top3  = basin_assignment[bname]["top3"]
        for m in top3:
            model_cell_mask[m][li, lj] = True
        n_models_per_cell[li, lj] = len(top3)

unique_counts = np.unique(n_models_per_cell[basin_id_grid >= 0])
print(f"Unique model-count values for Jordan cells : {unique_counts}")
print(f"  (all should be 3)")
print()
print("Model contribution (number of Jordan grid cells):")
for m in MODELS:
    n = int(model_cell_mask[m].sum())
    print(f"  {m:<22} : {n:>4} cells")

Unique model-count values for Jordan cells : [3]
  (all should be 3)

Model contribution (number of Jordan grid cells):
  CMCC-CM2-SR5           :  814 cells
  CNRM-ESM2-1            :   44 cells
  EC-Earth3-Veg          :    0 cells
  IPSL-CM6A-LR           :   97 cells
  MPI-ESM1-2-LR          :  829 cells
  NorESM2-MM             :  736 cells


## 7. Prepare basin_id Attribute String

In [7]:
basin_id_labels = "; ".join(
    f"{bi}={bname}" for bi, bname in enumerate(jordan_basin_names)
)
print("basin_id encoding:")
for bi, bname in enumerate(jordan_basin_names):
    n_cells = int((basin_id_grid == bi).sum())
    top3    = basin_assignment[bname]["top3"]
    print(f"  {bi:>2} = {bname:<32}  cells={n_cells:>4}  models={' | '.join(top3)}")
print()
print(f" -1 = outside Jordan domain  (cells={int((basin_id_grid < 0).sum()):,})")

basin_id encoding:
   0 = HAMMAD                            cells= 176  models=MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
   1 = YARMOUK (JORDAN)                  cells=  11  models=MPI-ESM1-2-LR | NorESM2-MM | CNRM-ESM2-1
   2 = J.VALLEY-YARMOUK TRIANGLE         cells=   0  models=NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
   3 = JORDAN VALLY (JORDAN)             cells=   5  models=MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
   4 = N.R.S.W                           cells=  11  models=NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
   5 = AZRAQ (JORDAN)                    cells= 113  models=MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
   6 = AMMAN ZARQA (JORDAN)              cells=  33  models=MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
   7 = S.R.S.W                           cells=   6  models=MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
   8 = MUJIB                             cells=  61  models=MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
   9 = D.S.R.S.W                         cells=  15  models=IPSL-CM6

## 8. Generate the Six Jordan-Wide NetCDF Files

In [8]:
all_times_pd = pd.DatetimeIndex(all_times)

model_assignment_str = " || ".join(
    f"{b}: {' + '.join(basin_assignment[b]['top3'])}"
    for b in jordan_basin_names
)

for period_label, y_start, y_end in PERIODS:
    out_filename = f"Jordan_3model_ensemble_ssp585_{period_label}.nc"
    out_path     = OUTPUT_DIR / out_filename

    if out_path.exists():
        print(f"[SKIP] {out_filename} already exists.")
        continue

    time_mask    = (all_times_pd.year >= y_start) & (all_times_pd.year <= y_end)
    time_idxs    = np.where(time_mask)[0]
    period_times = all_times[time_idxs]
    n_t          = len(time_idxs)

    print(f"[PROCESSING] {out_filename}")
    print(f"  Period    : {y_start}–{y_end}  ({n_t:,} time steps)")
    t0 = _time.time()

    ens_sum      = np.zeros((n_t, n_lat, n_lon), dtype=np.float32)
    outside_mask = (basin_id_grid < 0)

    for model in MODELS:
        cell_mask = model_cell_mask[model]
        if not cell_mask.any():
            continue
        nc_path = MODEL_DIR / f"merged_{model}.nc"
        with xr.open_dataset(nc_path) as ds_m:
            pr_model = ds_m[PR_VAR].isel(time=time_idxs).values
        ens_sum[:, cell_mask] += pr_model[:, cell_mask]
        del pr_model
        print(f"  {model:<22} accumulated")

    ens_pr = np.where(
        outside_mask[np.newaxis, :, :],
        np.nan,
        ens_sum / 3.0
    ).astype(np.float32)
    del ens_sum

    pr_da = xr.DataArray(
        ens_pr,
        dims=["time", "lat", "lon"],
        coords={"time": period_times, "lat": lats, "lon": lons},
        name=PR_VAR,
        attrs={
            "standard_name" : "precipitation_flux",
            "long_name"     : "Basin-Specific 3-Model Ensemble Mean Precipitation",
            "units"         : "mm/day",
            "cell_methods"  : "time: mean",
        }
    )

    basin_id_da = xr.DataArray(
        basin_id_grid,
        dims=["lat", "lon"],
        coords={"lat": lats, "lon": lons},
        name="basin_id",
        attrs={
            "long_name"       : "Jordan hydrological basin index",
            "basin_id_labels" : basin_id_labels,
            "note"            : "-1 = outside Jordan domain",
        }
    )

    ds_out = xr.Dataset(
        {PR_VAR: pr_da, "basin_id": basin_id_da},
        attrs={
            "title"                  : f"Jordan-Wide Basin-Specific 3-Model Ensemble Precipitation {y_start}-{y_end}",
            "institution"            : "Jordan University of Science and Technology",
            "scenario"               : "SSP5-8.5",
            "period"                 : f"{y_start}-{y_end}",
            "domain"                 : "Jordan  (0.1deg x 0.1deg)",
            "original_data"          : "RICAAR SMHI-HCLIM-ALADIN-38, SMHI-MIdAS021",
            "ensemble_method"        : "Basin-specific top-3 model ensemble",
            "basin_model_assignments": model_assignment_str,
            "Conventions"            : "CF-1.7",
            "history"                : (
                f"Created {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
                f"Basin ranking from Hussien et al. composite evaluation framework"
            ),
        }
    )

    ds_out["lat"].attrs  = {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}
    ds_out["lon"].attrs  = {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}
    ds_out["time"].attrs = {"long_name": "time"}

    encoding = {
        PR_VAR: {
            "dtype"     : "float32",
            "zlib"      : True,
            "complevel" : 4,
            "_FillValue": np.float32(1e+20),
        },
        "basin_id": {
            "dtype"     : "int8",
            "zlib"      : True,
            "complevel" : 4,
            "_FillValue": np.int8(-1),
        },
    }

    ds_out.to_netcdf(out_path, encoding=encoding, format="NETCDF4")
    del ens_pr, ds_out

    elapsed_s = _time.time() - t0
    size_mb   = out_path.stat().st_size / 1024 / 1024
    print(f"  Saved     : {out_filename}  ({size_mb:.1f} MB)  [{elapsed_s:.0f}s]")
    print()

print("=" * 65)
print("ALL SIX FILES COMPLETE")
print("=" * 65)

[PROCESSING] Jordan_3model_ensemble_ssp585_1961_2070.nc
  Period    : 1961–2070  (40,177 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accumulated
  Saved     : Jordan_3model_ensemble_ssp585_1961_2070.nc  (89.0 MB)  [19s]

[PROCESSING] Jordan_3model_ensemble_ssp585_1995_2014.nc
  Period    : 1995–2014  (7,305 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accumulated
  Saved     : Jordan_3model_ensemble_ssp585_1995_2014.nc  (17.9 MB)  [3s]

[PROCESSING] Jordan_3model_ensemble_ssp585_2015_2020.nc
  Period    : 2015–2020  (2,192 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accum

## 9. Verify Output Files

In [9]:
nc_out_files = sorted(OUTPUT_DIR.glob("Jordan_3model_ensemble_ssp585_*.nc"))
print(f"Files found: {len(nc_out_files)}")
print()
print(f"{'File':<52} {'Period':>18} {'Steps':>7} {'Jordan cells':>13} {'pr mean':>10} {'Size MB':>8}")
print("-" * 115)

for f in nc_out_files:
    ds = xr.open_dataset(f)
    n_t      = len(ds["time"])
    t0_str   = str(pd.Timestamp(ds["time"].values[0]).date())
    t1_str   = str(pd.Timestamp(ds["time"].values[-1]).date())
    pr_vals  = ds[PR_VAR].values
    pr_valid = pr_vals[~np.isnan(pr_vals)]
    bid      = ds["basin_id"].values
    n_jordan = int((bid >= 0).sum())
    size_mb  = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<50} {t0_str} → {t1_str} {n_t:>7,} {n_jordan:>13,} {pr_valid.mean():>10.4f} {size_mb:>8.1f}")
    ds.close()

Files found: 6

File                                                             Period   Steps  Jordan cells    pr mean  Size MB
-------------------------------------------------------------------------------------------------------------------
  Jordan_3model_ensemble_ssp585_1961_2070.nc         1961-01-01 → 2070-12-31  40,177           840     0.3099     89.0
  Jordan_3model_ensemble_ssp585_1995_2014.nc         1995-01-01 → 2014-12-31   7,305           840     0.2853     17.9
  Jordan_3model_ensemble_ssp585_2015_2020.nc         2015-01-01 → 2020-12-31   2,192           840     0.3282      5.5
  Jordan_3model_ensemble_ssp585_2021_2040.nc         2021-01-01 → 2040-12-31   7,305           840     0.2939     18.0
  Jordan_3model_ensemble_ssp585_2041_2060.nc         2041-01-01 → 2060-12-31   7,305           840     0.3053     18.0
  Jordan_3model_ensemble_ssp585_2061_2070.nc         2061-01-01 → 2070-12-31   3,652           840     0.3182      9.0
